### Testing neo4j cleint and cypher queries

In [1]:
from neo4j import GraphDatabase
import json

import os
os.chdir("../..")

In [2]:
driver = GraphDatabase.driver("bolt://localhost:7687", auth=("neo4j", "neo4j"))

In [ ]:
def add_mark(name: str): 
    query = """
    CREATE (mark:Mark {name: $name})
    RETURN mark
    """

    with driver.session() as session:
        result = session.run(query, name = name)
        return result.single()

def get_marks():
    query = "MATCH (mark: Mark) RETURN mark.name As name"
    with driver.session() as session:
        return [record["name"] for record in session.run(query)]


<Record mark=<Node element_id='4:0bea032b-bcd8-40e9-abd6-f6979d626659:0' labels=frozenset({'Mark'}) properties={'name': 'Southampton'}>>

In [6]:
add_mark("Sothampton")
add_mark("Cowes")

<Record mark=<Node element_id='4:0bea032b-bcd8-40e9-abd6-f6979d626659:3' labels=frozenset({'Mark'}) properties={'name': 'Cowes'}>>

In [7]:
get_marks()

['Southampton', 'Cowes', 'Sothampton', 'Cowes']

In [ ]:
def create_layer(name):
    query = """CALL spatial.addLayer('{name}', 'wkt', 'geom');"""
    with driver.session() as session:
        result = session.run(query, name = name)
        return result.single()

create_layer("solent_generic_spatial_layer")
    

<Record node=<Node element_id='4:8c8db40b-9cbb-4ef7-adab-208c9aced5a9:3' labels=frozenset({'SpatialLayer'}) properties={'index_class': 'org.neo4j.gis.spatial.index.LayerRTreeIndex', 'geomencoder_config': 'geom', 'geomencoder': 'org.neo4j.gis.spatial.WKTGeometryEncoder', 'layer_class': 'org.neo4j.gis.spatial.EditableLayerImpl', 'layer': 'solent_generic_spatial_layer'}>>

In [10]:
def add_marks_to_spatial_layer(layer_name, marks):
    cypher = """
    UNWIND $marks AS m
    CREATE (n:Mark)
    SET
      n.name = m.name,
      n.type = m.type,
      n.category = m.category,
      n.light = m.light,
      n.geom = 'POINT(' + toString(m.coordinates[0]) + ' ' + toString(m.coordinates[1]) + ')'
    WITH collect(n) AS nodes
    CALL spatial.addNodes($layer_name, nodes)
    YIELD count
    RETURN count;
    """

    with driver.session() as session:
        result = session.run(
            cypher,
            layer_name=layer_name,
            marks=marks
        )
        added = result.single()["count"]

    driver.close()
    return added

with open("src/kelder_api/assets/marks.json") as map:
    marks = json.load(map)

add_marks_to_spatial_layer("solent_generic_spatial_layer",marks)

943

In [ ]:
def get_marks_within_distance(distance):
    query = f"""CALL spatial.withinDistance(
  'solent_generic_spatial_layer',
  point({{longitude:-1.11, latitude:50.81}}),
  {distance}
)
YIELD node
RETURN node.name, node.type, node.category;"""
    
    with driver.session() as session:
        return [record for record in session.run(query)]
    
get_marks_within_distance(500)
    

Received notification from DBMS server: <GqlStatusObject gql_status='01N52', status_description='warn: property key does not exist. The property `category` does not exist in database `neo4j`. Verify that the spelling is correct.', position=<SummaryInputPosition line=7, column=35, offset=160>, raw_classification='UNRECOGNIZED', classification=<NotificationClassification.UNRECOGNIZED: 'UNRECOGNIZED'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'UNRECOGNIZED', '_severity': 'WARNING', '_position': {'offset': 160, 'line': 7, 'column': 35}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "CALL spatial.withinDistance(\n  'solent_generic_spatial_layer',\n  point({longitude:-1.11, latitude:50.81}),\n  500\n)\nYIELD node\nRETURN node.name, node.type, node.category;"


[<Record node.name='99' node.type='lateral_mark' node.category=None>,
 <Record node.name='98' node.type='cardinal_mark' node.category=None>,
 <Record node.name='101' node.type='cardinal_mark' node.category=None>,
 <Record node.name='100' node.type='lateral_mark' node.category=None>,
 <Record node.name=None node.type='lateral_mark' node.category=None>,
 <Record node.name=None node.type='special_purpose' node.category=None>,
 <Record node.name=None node.type='special_purpose' node.category=None>,
 <Record node.name='102' node.type='lateral_mark' node.category=None>,
 <Record node.name=None node.type='special_purpose' node.category=None>,
 <Record node.name=None node.type='special_purpose' node.category=None>,
 <Record node.name=None node.type='special_purpose' node.category=None>,
 <Record node.name='103' node.type='lateral_mark' node.category=None>,
 <Record node.name=None node.type='special_purpose' node.category=None>,
 <Record node.name='104' node.type='lateral_mark' node.category=No

In [4]:
DELETE_ALL_NODES = """
MATCH (n) DETACH DELETE n
"""
def delete_nodes():
    with driver.session() as session:
        result = session.run(
            DELETE_ALL_NODES,
        )

        driver.close()
        return True
    
delete_nodes()

True